02 — Nettoyage et préparation des données Fifa
Objectif

Préparer un jeu de données regroupant les résultats des Coupes du monde de football de 1930 à 2022.

Les étapes prévues sont :

    Importer et assembler les fichiers CSV des différentes éditions.

    Vérifier la qualité et la cohérence des données.

    Traiter les éventuelles anomalies.

    Harmoniser les noms des équipes.

    Identifier les pays organisateurs de chaque édition.

Ce jeu de données préparé servira ensuite à analyser les performances des pays hôtes et à étudier leur éventuel avantage sportif.

In [1]:
import pandas as pd
import glob
import os

In [2]:
fichiers_csv = glob.glob("../data/FIFA - *.csv")

print(len(fichiers_csv))

22


In [3]:
liste_df = []

for fichier in fichiers_csv:
    df_temp = pd.read_csv(fichier)

    annee = int(os.path.basename(fichier).replace("FIFA - ", "").replace(".csv", ""))

    df_temp["Year"] = annee

    liste_df.append(df_temp)

In [4]:
df_fifa = pd.concat(liste_df, ignore_index=True)

In [5]:
print(df_fifa.shape)
print(df_fifa["Year"].nunique())

(489, 11)
22


In [6]:
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489 entries, 0 to 488
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Position         489 non-null    int64 
 1   Team             489 non-null    object
 2   Games Played     489 non-null    int64 
 3   Win              489 non-null    int64 
 4   Draw             489 non-null    int64 
 5   Loss             489 non-null    int64 
 6   Goals For        489 non-null    int64 
 7   Goals Against    489 non-null    int64 
 8   Goal Difference  489 non-null    object
 9   Points           489 non-null    int64 
 10  Year             489 non-null    int64 
dtypes: int64(9), object(2)
memory usage: 42.1+ KB


In [7]:
df_fifa["Goal Difference"] = (
    df_fifa["Goal Difference"]
    .astype(str)
    .str.replace("−", "-", regex=False)
    .astype(int)
)

On s'est aperçu lors de la phase d'exploration que pour la colonne Goal difference, les différences de buts négatives étaient indiquées par un tiret non mathématique ce qui passait la colonne en type objet. On remplace donc les tirets par des "moins" mathématiques.

In [8]:
df_fifa["Goal Difference"].dtype

dtype('int64')

On vérifie ensuite que pour chaque équipe, le nombre de matchs joués corresponde bien à la somme des victoires, des matchs nuls et des défaites.

In [9]:
df_fifa["Total Results"] = df_fifa["Win"] + df_fifa["Draw"] + df_fifa["Loss"]

df_fifa[df_fifa["Games Played"] != df_fifa["Total Results"]]

,Position,Team,Games Played,Win,Draw,Loss,Goals For,Goals Against,Goal Difference,Points,Year,Total Results


Non n'avons plus besoin de la colonne Total results, nous pouvons donc la supprimer, Games played suffit. 

In [10]:
df_fifa = df_fifa.drop(columns=["Total Results"])

On vérifie s'il y a des lignes strictement identiques.

In [11]:
df_fifa.duplicated().sum()

np.int64(0)

Il faut maintenant régler un détail important pour notre problématique : harmoniser les noms des pays au fil des 90 ans. Pour cela, on se base sur l'héritage des fédérations officiellement retenu par la FIFA. Par exemple, l'Allemagne depuis 1990 est l'héritière de la RFA, la Russie de l'URSS, la Serbie de la Yougoslavie, ou encore la République Tchèque de la Tchécoslovaquie.

In [12]:
sorted(df_fifa["Team"].unique())

['Algeria',
 'Angola',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Brazil',
 'Bulgaria',
 'Bulgaria**',
 'Cameroon',
 'Canada',
 'Chile',
 'China PR',
 'Colombia',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Czechoslovakia',
 'Denmark',
 'Dutch East Indies',
 'East Germany',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'FR Yugoslavia',
 'France',
 'Germany',
 'Ghana',
 'Greece',
 'Haiti',
 'Honduras',
 'Hungary',
 'Iceland',
 'Iran',
 'Iraq',
 'Israel*',
 'Italy',
 'Ivory Coast',
 'Jamaica',
 'Japan',
 'Kuwait',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Nigeria',
 'North Korea',
 'Northern Ireland',
 'Norway',
 'Panama',
 'Paraguay',
 'Peru',
 'Poland',
 'Portugal',
 'Qatar',
 'Republic of Ireland',
 'Romania',
 'Russia',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'Serbia',
 'Serbia and Montenegro',
 'Slovakia',
 'Slovenia',
 'South Africa',
 'South Korea',
 'Soviet Union',
 'Spain',
 'Sweden',
 'Switzer

In [13]:
for team in sorted(df_fifa["Team"].unique()):
    print(team)

Algeria
Angola
Argentina
Australia
Austria
Belgium
Bolivia
Bosnia and Herzegovina
Brazil
Bulgaria
Bulgaria**
Cameroon
Canada
Chile
China PR
Colombia
Costa Rica
Croatia
Cuba
Czech Republic
Czechoslovakia
Denmark
Dutch East Indies
East Germany
Ecuador
Egypt
El Salvador
England
FR Yugoslavia
France
Germany
Ghana
Greece
Haiti
Honduras
Hungary
Iceland
Iran
Iraq
Israel*
Italy
Ivory Coast
Jamaica
Japan
Kuwait
Mexico
Morocco
Netherlands
New Zealand
Nigeria
North Korea
Northern Ireland
Norway
Panama
Paraguay
Peru
Poland
Portugal
Qatar
Republic of Ireland
Romania
Russia
Saudi Arabia
Scotland
Senegal
Serbia
Serbia and Montenegro
Slovakia
Slovenia
South Africa
South Korea
Soviet Union
Spain
Sweden
Switzerland
Togo
Trinidad and Tobago
Tunisia
Turkey
USA
Ukraine
United Arab Emirates
United States
Uruguay
Wales
West Germany
Yugoslavia
Zaire


In [14]:
df_fifa[df_fifa["Team"].str.contains(r"\*")][["Year", "Team"]]

,Year,Team
132,1970,Israel*
133,1970,Bulgaria**


In [15]:
df_fifa["Team"] = df_fifa["Team"].str.replace("*", "", regex=False).str.strip()

In [16]:
df_fifa[df_fifa["Year"] == 1970][["Year", "Team"]]

,Year,Team
121,1970,Brazil
122,1970,Italy
123,1970,West Germany
124,1970,Uruguay
125,1970,Soviet Union
126,1970,Mexico
127,1970,Peru
128,1970,England
129,1970,Sweden
130,1970,Belgium


In [17]:
df_fifa[df_fifa["Team"].isin([
    "Germany", "West Germany", "East Germany",
    "Soviet Union", "Russia",
    "Czechoslovakia", "Czech Republic", "Slovakia",
    "Yugoslavia", "FR Yugoslavia",
    "Serbia and Montenegro", "Serbia"
])][["Year", "Team"]].sort_values(["Year", "Team"])

,Year,Team
3,1930,Yugoslavia
14,1934,Czechoslovakia
15,1934,Germany
33,1938,Czechoslovakia
38,1938,Germany
48,1950,Yugoslavia
70,1954,Czechoslovakia
57,1954,West Germany
64,1954,Yugoslavia
81,1958,Czechoslovakia


Méthodologie d’harmonisation des équipes nationales :

Le dataset regroupe les résultats des Coupes du monde de 1930 à 2022, or certaines équipes nationales ont changé de nom à la suite d'évolutions politiques.

Afin de permettre des analyses cohérentes dans le temps, nous créons une colonne Team_Harmonized à partir de la colonne originale Team.

Le principe :

Team conserve le nom de l'équipe tel qu'il apparaît dans les données sources, après suppression des caractères parasites.

Team_Harmonized attribue à chaque équipe un nom de référence commun, selon la continuité sportive officielle de la FIFA.

Les équipes dont le nom n'est pas concerné par une harmonisation conservent leur nom d'origine.

Les changements historiques ne sont pas tous traités de la même manière : une nouvelle équipe nationale issue de la dissolution d'un État n'est pas automatiquement assimilée à une autre équipe issue de cette même dissolution.

C'est un choix méthodologique pour faciliter les comparaisons dans le temps. Elle devra être prise en compte lors de l'interprétation des résultats.

Règles d'harmonisation retenues : 

Nom d'origine (Team)

Nom harmonisé (Team_Harmonized)

Liste des harmonisations : 

West Germany, Germany  =  Germany

Soviet Union, Russia = Russia

Yugoslavia, FR Yugoslavia, Serbia and Montenegro, Serbia = Serbia

Czechoslovakia, Czech Republic = Czech Republic

Dutch East Indies = Indonesia

Zaire = DR Congo

USA = United States (simple harmonisation de la dénomination)

Cas particulier : East Germany reste distinct de Germany. Le regroupement retenu pour l'Allemagne de l'Ouest ne s'applique donc pas automatiquement à l'Allemagne de l'Est. Idem pour la Slovaquie qui reste distinct de la Tchécoslovaquie, ou l'Ukraine de l'URSS.

Conséquences pour l'analyse :

Cette harmonisation permet de regrouper certaines performances historiques sous un même nom de référence et de faciliter les comparaisons entre éditions de la Coupe du monde.

Elle implique toutefois des choix d'attribution historique : les résultats regroupés sous un même nom ne correspondent pas nécessairement à une seule équipe nationale ayant existé sous une identité inchangée pendant toute la période.

La colonne Team sera donc conservée afin de préserver la traçabilité des données sources. La colonne Team_Harmonized sera utilisée pour les analyses au long cours qui nécessitent ces regroupements.

In [18]:
df_fifa["Team_Harmonized"] = df_fifa["Team"]
correspondances = {
    "West Germany": "Germany",
    "Soviet Union": "Russia",
    "Yugoslavia": "Serbia",
    "FR Yugoslavia": "Serbia",
    "Serbia and Montenegro": "Serbia",
    "Czechoslovakia": "Czech Republic",
    "Dutch East Indies": "Indonesia",
    "Zaire": "DR Congo",
    "USA": "United States"
}
df_fifa["Team_Harmonized"] = (
    df_fifa["Team_Harmonized"].replace(correspondances)
)


In [19]:
df_fifa[
    df_fifa["Team"] != df_fifa["Team_Harmonized"]
][["Year", "Team", "Team_Harmonized"]].sort_values(
    ["Year", "Team"]
)


,Year,Team,Team_Harmonized
3,1930,Yugoslavia,Serbia
14,1934,Czechoslovakia,Czech Republic
33,1938,Czechoslovakia,Czech Republic
43,1938,Dutch East Indies,Indonesia
48,1950,Yugoslavia,Serbia
70,1954,Czechoslovakia,Czech Republic
57,1954,West Germany,Germany
64,1954,Yugoslavia,Serbia
81,1958,Czechoslovakia,Czech Republic
78,1958,Soviet Union,Russia


In [20]:
df_fifa[
    df_fifa["Year"] == 2018
][["Year", "Team", "Team_Harmonized"]].sort_values("Team")

,Year,Team,Team_Harmonized
439,2018,Argentina,Argentina
451,2018,Australia,Australia
427,2018,Belgium,Belgium
430,2018,Brazil,Brazil
433,2018,Colombia,Colombia
452,2018,Costa Rica,Costa Rica
426,2018,Croatia,Croatia
435,2018,Denmark,Denmark
455,2018,Egypt,Egypt
428,2018,England,England


In [21]:
df_fifa[
    df_fifa["Year"] == 2006
][["Year", "Team", "Team_Harmonized"]].sort_values("Team")

,Year,Team,Team_Harmonized
351,2006,Angola,Angola
334,2006,Argentina,Argentina
344,2006,Australia,Australia
333,2006,Brazil,Brazil
359,2006,Costa Rica,Costa Rica
350,2006,Croatia,Croatia
348,2006,Czech Republic,Czech Republic
340,2006,Ecuador,Ecuador
335,2006,England,England
330,2006,France,France


In [23]:
df_fifa[
    df_fifa["Year"] == 1994
][["Year", "Team", "Team_Harmonized"]].sort_values("Team")

,Year,Team,Team_Harmonized
250,1994,Argentina,Argentina
251,1994,Belgium,Belgium
261,1994,Bolivia,Bolivia
241,1994,Brazil,Brazil
244,1994,Bulgaria,Bulgaria
262,1994,Cameroon,Cameroon
259,1994,Colombia,Colombia
245,1994,Germany,Germany
264,1994,Greece,Greece
242,1994,Italy,Italy


In [24]:
noms_attendus = df_fifa["Team"].replace(correspondances)
erreurs = df_fifa[
    noms_attendus != df_fifa["Team_Harmonized"]
][["Year", "Team", "Team_Harmonized"]]
print("Nombre d'erreurs :", len(erreurs))
display(erreurs)

Nombre d'erreurs : 0


,Year,Team,Team_Harmonized
